# Embeddings

This section generates dense vector representations for all normalized questions using a sentence-transformers bi-encoder, allowing us to capture semantic paraphrases that the lexical MinHash baseline misses. To bypass the impossible computational cost of an exact all-pairs comparison, we build a FAISS HNSW index to efficiently retrieve the top 10-20 nearest neighbors for each question based on cosine similarity.

In [4]:
import pandas as pd
from sentence_transformers import SentenceTransformer, InputExample, losses
import numpy as np
import faiss
from torch.utils.data import DataLoader
import time
from scipy.spatial.distance import cdist
import os
import math
import random

C:\Users\Alberto\AppData\Local\Temp\ipykernel_18580\2765298263.py:2: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses


In [5]:
print(os.getcwd())

df = pd.read_parquet('../data/processed/questions.parquet')
texts = df['text_norm'].fillna("").tolist()
qids = df['qid'].tolist()

model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(texts, batch_size=256, show_progress_bar=True, normalize_embeddings=True)

np.save('embeddings.npy', embeddings)

c:\Users\Alberto\Desktop\Computational Tools\quora-question-pairs\notebooks


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/2099 [00:00<?, ?it/s]

In [6]:
d = embeddings.shape[1]
index_hnsw = faiss.IndexHNSWFlat(d, 32, faiss.METRIC_INNER_PRODUCT)
index_hnsw.add(embeddings)

k = 20
threshold = 0.80

scores, indices = index_hnsw.search(embeddings, k)

candidates = []
for i in range(len(qids)):
    for j in range(1, k):
        score = float(scores[i][j])
        if score >= threshold:
            candidates.append({
                'qid_a': qids[i],
                'qid_b': qids[indices[i][j]],
                'score': score,
                'source': 'emb'
            })

df_candidates = pd.DataFrame(candidates)
df_candidates.to_parquet('candidates_emb.parquet', index=False)

In [7]:
sample_size = min(1000, len(embeddings))
sample_queries = embeddings[:sample_size]

index_exact = faiss.IndexFlatIP(d)
index_exact.add(embeddings)

exact_scores, exact_indices = index_exact.search(sample_queries, k)
hnsw_scores, hnsw_indices = index_hnsw.search(sample_queries, k)

hits = 0
for i in range(sample_size):
    exact_set = set(exact_indices[i])
    hnsw_set = set(hnsw_indices[i])
    hits += len(exact_set.intersection(hnsw_set))

recall = hits / (sample_size * k)
print(f"HNSW Recall: {recall:.4f}")

HNSW Recall: 0.9615


# Educational

In [5]:
df = pd.read_parquet('../data/processed/questions.parquet')
qids = df['qid'].tolist()

embeddings = np.load('embeddings.npy')

In [6]:

class Node:
    def __init__(self, vector, node_id, level):
        self.vector = vector
        self.node_id = node_id
        self.level = level
        self.connections = {}

def euclidean_distance(v1, v2):
    return np.linalg.norm(v1 - v2)

class SimpleHNSW:
    def __init__(self, m=5, m_max=16, m_max0=32, m_l=0.36):
        self.m = m
        self.m_max = m_max
        self.m_max0 = m_max0
        self.m_l = m_l
        self.enter_point = None
        self.max_level = -1
        self.nodes = {}

    def get_random_level(self):
        return math.floor(-math.log(random.random()) * self.m_l)

    def search_layer(self, query, entry_node, ef, layer):
        visited = {entry_node.node_id}
        candidates = [(euclidean_distance(query, entry_node.vector), entry_node)]
        nearest = candidates.copy()
        
        while candidates:
            candidates.sort(key=lambda x: x[0])
            c_dist, c_node = candidates.pop(0)
            
            nearest.sort(key=lambda x: x[0])
            if c_dist > nearest[-1][0] and len(nearest) == ef:
                break
                
            for neighbor_id in c_node.connections.get(layer, []):
                if neighbor_id not in visited:
                    visited.add(neighbor_id)
                    neighbor_node = self.nodes[neighbor_id]
                    dist = euclidean_distance(query, neighbor_node.vector)
                    
                    if len(nearest) < ef or dist < nearest[-1][0]:
                        candidates.append((dist, neighbor_node))
                        nearest.append((dist, neighbor_node))
                        nearest.sort(key=lambda x: x[0])
                        if len(nearest) > ef:
                            nearest.pop()
                            
        return nearest

    def insert(self, vector, node_id):
        level = self.get_random_level()
        new_node = Node(vector, node_id, level)
        self.nodes[node_id] = new_node
        
        if self.enter_point is None:
            self.enter_point = new_node
            self.max_level = level
            return
            
        curr_node = self.enter_point
        for l in range(self.max_level, level, -1):
            nearest = self.search_layer(vector, curr_node, 1, l)
            curr_node = nearest[0][1]
            
        for l in range(min(level, self.max_level), -1, -1):
            nearest = self.search_layer(vector, curr_node, self.m, l)
            new_node.connections[l] = []
            for dist, neighbor in nearest:
                new_node.connections[l].append(neighbor.node_id)
                if l not in neighbor.connections:
                    neighbor.connections[l] = []
                neighbor.connections[l].append(node_id)
            curr_node = nearest[0][1]
            
        if level > self.max_level:
            self.max_level = level
            self.enter_point = new_node

In [7]:
custom_hnsw = SimpleHNSW()
sample_subset = embeddings[:1000]

start_time = time.perf_counter()

for i, vec in enumerate(sample_subset):
    custom_hnsw.insert(vec, i)
    
print(f"Custom HNSW built for 1000 vectors in {time.perf_counter() - start_time:.2f}s")

Custom HNSW built for 1000 vectors in 0.36s


In [11]:
import faiss

d = embeddings.shape[1]
mini_faiss_index = faiss.IndexHNSWFlat(d, 32, faiss.METRIC_INNER_PRODUCT)
mini_faiss_index.add(sample_subset)

k = 20
mini_hnsw_scores, mini_hnsw_indices = mini_faiss_index.search(sample_subset, k)

custom_hits = 0

for i in range(len(sample_subset)):
    query = sample_subset[i]
    
    custom_nearest = custom_hnsw.search_layer(query, custom_hnsw.enter_point, k, 0)
    custom_indices = {node.node_id for dist, node in custom_nearest}
    
    faiss_set = set(mini_hnsw_indices[i])
    
    custom_hits += len(custom_indices.intersection(faiss_set))

custom_recall = custom_hits / (len(sample_subset) * k)
print(f"Custom HNSW Recall vs Mini-FAISS (k={k}): {custom_recall:.4f}")

Custom HNSW Recall vs Mini-FAISS (k=20): 0.7305
